# 05 - Azure AI Search: Setup with Bicep

Goal: Deploy Azure AI Search, Storage Account, and RBAC using Infrastructure as Code (Bicep), then create indices with Python.

**What this notebook does:**
1. Deploys Bicep template to create:
   - Azure AI Search service with managed identity
   - Storage account with blob containers (data source for indexing)
   - RBAC assignments (Search → Storage, Blueprint → Search)
2. Creates search indices using Python SDK
3. Seeds sample documents (2-3 per index)

**Prerequisites:**
- `.env` file with `AGENT_BLUEPRINT_PRINCIPAL_ID` (other vars have sensible defaults)
- Admin permissions in your Azure subscription (you'll be prompted to log in interactively)

▶️ Click `Run All` to execute all steps sequentially, or execute them `Step by Step`...

In [ ]:
import os
import json
import subprocess
from dotenv import load_dotenv
import utils
from azure.identity import InteractiveBrowserCredential

load_dotenv()

# Fix PATH for Azure CLI - common installation locations
az_paths = [
    '/usr/local/bin',
    '/opt/homebrew/bin',
    '/usr/bin',
    os.path.expanduser('~/bin')
]
for path in az_paths:
    if path not in os.environ['PATH']:
        os.environ['PATH'] = f"{path}:{os.environ['PATH']}"

subscription_id = os.getenv('AZURE_SUBSCRIPTION_ID')
# Load configuration from .env
resource_group_name = os.getenv('AZURE_RESOURCE_GROUP', 'rg-agent-blueprint-demo')
resource_group_location = os.getenv('AZURE_LOCATION', 'eastus')
search_service_name = os.getenv('AZURE_SEARCH_SERVICE_NAME', '')
storage_account_name = os.getenv('AZURE_STORAGE_ACCOUNT_NAME', '')
blueprint_principal_id = os.getenv('AGENT_BLUEPRINT_PRINCIPAL_ID')

# Initialize interactive browser credential for admin login
# This gets a token that az CLI can use
credential = InteractiveBrowserCredential()

# Validate required variables
required = {
    'AGENT_BLUEPRINT_PRINCIPAL_ID': blueprint_principal_id
}

missing = [k for k, v in required.items() if not v]
if missing:
    raise ValueError(f"❌ Missing in .env: {', '.join(missing)}")

utils.print_ok('Notebook initialized')
utils.print_info(f"Resource Group: {resource_group_name}")
utils.print_info(f"Location: {resource_group_location}")
utils.print_info(f"Search Service: {search_service_name}")
utils.print_info(f"Storage Account: {storage_account_name}")
utils.print_info(f"Blueprint Principal: {blueprint_principal_id[:8]}...")

In [ ]:
# Verify Azure CLI is accessible and authenticate
test_az = utils.run("az --version", print_command_to_run=False, print_output=False)
if not test_az.success:
    utils.print_error("Azure CLI not found. Run in terminal: which az")
    utils.print_info("Then add that path to cell 1 above")
    raise RuntimeError("Cannot find az CLI")

utils.print_ok("Azure CLI is accessible")

# Authenticate with interactive browser login
try:
    token = credential.get_token('https://management.azure.com/.default')
    utils.print_ok("Azure authentication successful")
    utils.print_info(f"Authenticated with your Azure deployment identity credentials")
    
    # Get subscription ID from az account
    if not subscription_id:
        output = utils.run("az account show --query id --output tsv", print_command_to_run=False)
        if output.success:    
            subscription_id = output.text.strip()    
            utils.print_info(f"Subscription: {subscription_id}")
        else:
            utils.print_warning("Could not retrieve subscription from az CLI, but will continue")
except Exception as e:
    utils.print_error(f"Azure authentication failed: {str(e)}")
    utils.print_info("Interactive browser login is required. Check your browser for the login prompt.")
    raise

<a id='1'></a>
## 1️⃣ Create deployment using 🦾 Bicep

This lab uses [Bicep](https://learn.microsoft.com/azure/azure-resource-manager/bicep/overview?tabs=bicep) to declaratively define all resources. Change the parameters or the [main.bicep](search-setup.bicep) directly to try different configurations.

**Defaults:** The templates now generate deterministic, unique names (using `uniqueString`) for the resource group (subscription scope), search service, and storage account, and default the location to the resource group location. Leave the related `.env` entries empty to let the template pick those safe defaults; set them only when you need specific names/regions.

In [ ]:
# Create resource group if it doesn't exist
try:
    output = utils.run(
        f"az group create --name {resource_group_name} --location {resource_group_location} --tags source=agent365",
        print_command_to_run=False
    )
    if output.success:
        utils.print_ok(f"Resource group '{resource_group_name}' ready")
except Exception as e:
    utils.print_error(f"Failed to create resource group: {str(e)}")
    raise

# Define Bicep deployment parameters
deployment_params = {
    "blueprintPrincipalId": {"value": blueprint_principal_id}
}

if search_service_name:
    deployment_params["searchServiceName"] = {"value": search_service_name}
if storage_account_name:
    deployment_params["storageAccountName"] = {"value": storage_account_name}
if resource_group_location:
    deployment_params["location"] = {"value": resource_group_location}

# Write parameters to temp file
with open('deployment-params.json', 'w') as f:
    json.dump({"$schema": "https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#", 
               "contentVersion": "1.0.0.0", "parameters": deployment_params}, f, indent=2)

# Deploy Bicep template
deployment_name = "search-setup"
output = utils.run(
    f"az deployment group create --name {deployment_name} --resource-group {resource_group_name} --template-file search-resources.bicep --parameters deployment-params.json",
    f"Deployment '{deployment_name}' succeeded",
    f"Deployment '{deployment_name}' failed"
)

In [ ]:
# Retrieve deployment outputs
output = utils.run(
    f"az deployment group show --name {deployment_name} -g {resource_group_name}",
    f"Retrieved deployment: {deployment_name}",
    f"Failed to retrieve deployment: {deployment_name}"
)

if output.success and output.json_data:
    endpoint = utils.get_deployment_output(output, 'searchEndpoint', 'Search Endpoint')
    search_principal_id = utils.get_deployment_output(output, 'searchPrincipalId', 'Search Managed Identity')
    search_service_name = utils.get_deployment_output(output, 'searchServiceName', 'Search Service Name')
    storage_endpoint = utils.get_deployment_output(output, 'storageEndpoint', 'Storage Endpoint')
    us_container = utils.get_deployment_output(output, 'usContainerName', 'US Container')
    apac_container = utils.get_deployment_output(output, 'apacContainerName', 'APAC Container')

<a id='2'></a>
## 2️⃣ Get Search Admin Key

Retrieve the admin key for data plane operations (creating indices, uploading documents).

In [ ]:
# Retrieve the Search admin key
output = utils.run(
    f"az search admin-key show --resource-group {resource_group_name} --service-name {search_service_name} --query primaryKey --output tsv",
    "Admin key retrieved",
    "Failed to retrieve admin key"
)

if output.success:
    api_key = output.text.strip()
    utils.print_info(f"Key: ****{api_key[-4:]}")

<a id='3'></a>
## 3️⃣ Create Search Indices

Now that the search service exists, let's create the indices for our demo.

In [ ]:
# Create two indices: agents-us and agents-apac
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex, SimpleField, SearchableField, CorsOptions
)

index_client = SearchIndexClient(endpoint=endpoint, credential=AzureKeyCredential(api_key))

def ensure_index(name: str):
    fields = [
        SimpleField(name='id', type='Edm.String', key=True),
        SearchableField(name='title', type='Edm.String'),
        SearchableField(name='content', type='Edm.String'),
        SimpleField(name='region', type='Edm.String')
    ]
    index = SearchIndex(name=name, fields=fields, cors_options=CorsOptions(allowed_origins=['*']))
    try:
        index_client.create_index(index)
        utils.print_ok(f'Created index: {name}')
    except Exception as e:
        utils.print_warning(f'Index {name} may already exist: {e}')

ensure_index('agents-us')
ensure_index('agents-apac')

In [ ]:
# Upload 2-3 docs per index
from azure.search.documents import SearchClient

def seed_index(name: str, docs: list):
    client = SearchClient(endpoint=endpoint, index_name=name, credential=AzureKeyCredential(api_key))
    result = client.upload_documents(documents=docs)
    succeeded = sum(1 for r in result if r.succeeded)
    utils.print_ok(f'Uploaded {succeeded}/{len(docs)} docs to {name}')

seed_index('agents-us', [
    {'id': 'us-1', 'title': 'US FAQ', 'content': 'Shipping policy for US region', 'region': 'US'},
    {'id': 'us-2', 'title': 'US Returns', 'content': 'Return window and process in US', 'region': 'US'},
    {'id': 'us-3', 'title': 'US Taxes', 'content': 'Sales tax handling for US orders', 'region': 'US'},
])

seed_index('agents-apac', [
    {'id': 'apac-1', 'title': 'APAC FAQ', 'content': 'Shipping policy for APAC region', 'region': 'APAC'},
    {'id': 'apac-2', 'title': 'APAC Returns', 'content': 'Return window and process in APAC', 'region': 'APAC'},
    {'id': 'apac-3', 'title': 'APAC Taxes', 'content': 'GST/VAT handling for APAC orders', 'region': 'APAC'},
])

<a id='clean'></a>
## 🗑️ Clean up resources

When you're finished with the lab, you should remove all your deployed resources from Azure to avoid extra charges and keep your Azure subscription uncluttered.

```python
output = utils.run(
    f"az group delete --name {resource_group_name} -y --no-wait",
    f"Resource group '{resource_group_name}' deletion initiated",
    f"Failed to delete resource group '{resource_group_name}'"
)
```

## Summary

✅ **What was created:**
- Azure AI Search service with managed identity
- Storage account with blob containers (`agents-us-data`, `agents-apac-data`)
- Search indices with sample documents
- RBAC: Search service → Storage (for data source indexing)
- RBAC: Blueprint principal → Search (service-level reader access)

**Next steps:**
- **[06-search-rbac-demo.ipynb](./06-search-rbac-demo.ipynb)**: Test index-scoped RBAC and selective agent access